
先明确三个非常重要的结论：

1. **DDPM 首先是一个潜变量概率生成模型。** 它从最大化数据对数似然出发，通过变分推断得到 ELBO，再将 ELBO 分解成一系列 KL 散度。
2. **预测噪声的 MSE 不是凭空设计的。** 在反向转移采用固定方差高斯分布的条件下，每一步的 KL 可以转化为后验均值的平方误差，再通过参数重写转化为噪声预测误差。
3. **论文最终效果最好的 $L_{\mathrm{simple}}$ 并不严格等于原始负 ELBO。** 作者去掉了不同时间步的理论权重，相当于重新加权训练目标，实验发现这能显著改善生成质量。[arXiv](https://arxiv.org/pdf/2006.11239?utm_source=chatgpt.com)

这三个结论贯穿整篇论文，也是理解 DDPM 与 NCSN、VAE、Score Matching 之间关系的关键。

---

# DDPM（2020）完整理论学习笔记

**论文名称：** Denoising Diffusion Probabilistic Models  
**作者：** Jonathan Ho、Ajay Jain、Pieter Abbeel  
**会议：** NeurIPS 2020  
**论文：** [arXiv:2006.11239](https://arxiv.org/pdf/2006.11239)

## 一、论文整体思想与数学路线

DDPM 的目标仍然是生成建模。

假设真实数据来自未知分布：

$$
x_0\sim q_{\mathrm{data}}(x_0)
$$

我们希望构造参数化概率模型：

$$
p_\theta(x_0)
$$

使其尽可能接近真实数据分布。

从最大似然估计的角度，训练目标是：

$$
\boxed{
\max_\theta
\mathbb E_{x_0\sim q_{\mathrm{data}}}
[\log p_\theta(x_0)]
}
$$

等价于最小化：

$$
\boxed{
\min_\theta
\mathbb E_{x_0\sim q_{\mathrm{data}}}
[-\log p_\theta(x_0)]
}
$$

但 DDPM 并不直接对 $p_\theta(x_0)$ 建模。

它引入大量中间潜变量：

$$
x_1,x_2,\ldots,x_T
$$

然后构造：

$$
\boxed{
p_\theta(x_0)
=
\int p_\theta(x_{0:T})\,dx_{1:T}
}
$$

其中：

$$
x_{0:T}=(x_0,x_1,\ldots,x_T)
$$

整个理论路线是：

$$
\boxed{
\begin{aligned}
&\text{最大化数据对数似然}\\
&\Downarrow\\
&\text{引入潜变量 }x_{1:T}\\
&\Downarrow\\
&\text{构造前向高斯马尔可夫链 }q\\
&\Downarrow\\
&\text{构造反向生成马尔可夫链 }p_\theta\\
&\Downarrow\\
&\text{利用 Jensen 不等式推导 ELBO}\\
&\Downarrow\\
&\text{将 ELBO 分解为多个 KL 项}\\
&\Downarrow\\
&\text{解析计算高斯后验 }q(x_{t-1}\mid x_t,x_0)\\
&\Downarrow\\
&\text{高斯 KL 转化为均值预测 MSE}\\
&\Downarrow\\
&\text{将反向均值参数化为噪声预测}\\
&\Downarrow\\
&\text{得到加权噪声预测损失}\\
&\Downarrow\\
&\text{去除权重得到 }L_{\mathrm{simple}}\\
&\Downarrow\\
&\text{训练网络并进行反向马尔可夫采样}
\end{aligned}
}
$$

下面严格按照这条路线展开。

---

# 第一部分：DDPM 的概率模型基础

## 二、DDPM 与 VAE 的关系

### 2.1 先回顾普通 VAE

普通 VAE 引入潜变量：

$$
z\sim p(z)
$$

然后通过解码器：

$$
p_\theta(x\mid z)
$$

生成数据。

因此：

$$
\boxed{
p_\theta(x,z)=p(z)p_\theta(x\mid z)
}
$$

数据边缘似然为：

$$
p_\theta(x)
=
\int p(z)p_\theta(x\mid z)\,dz
$$

由于这个积分通常难以直接计算，所以引入近似后验：

$$
q_\phi(z\mid x)
$$

通过变分推断优化 ELBO。

### 2.2 DDPM 相当于一个具有特殊结构的多层潜变量模型

DDPM 引入：

$$
x_1,\ldots,x_T
$$

作为潜变量。

其反向生成联合分布为：

$$
\boxed{
p_\theta(x_{0:T})
=
p(x_T)
\prod_{t=1}^{T}
p_\theta(x_{t-1}\mid x_t)
}
\tag{1}
$$

其中：

$$
\boxed{
p(x_T)=\mathcal N(0,I)
}
$$

而每一步反向转移定义为：

$$
\boxed{
p_\theta(x_{t-1}\mid x_t)
=
\mathcal N
\left(
x_{t-1};
\mu_\theta(x_t,t),
\Sigma_\theta(x_t,t)
\right)
}
$$

因此，生成过程为：

$$
x_T\sim\mathcal N(0,I)
$$

$$
x_{T-1}\sim p_\theta(x_{T-1}\mid x_T)
$$

一直到：

$$
x_0\sim p_\theta(x_0\mid x_1)
$$

从概率模型角度看，DDPM 与 VAE 非常相似：

- VAE 使用一个或少量潜变量。
- DDPM 使用大量同维度的潜变量。
- VAE 通常学习编码器和解码器。
- DDPM 的前向加噪过程通常固定，只学习反向生成过程。

因此，可以将 DDPM 理解为一种**具有固定前向推断过程和多层马尔可夫潜变量结构的生成模型**。

---

## 三、前向扩散过程

DDPM 定义前向过程：

$$
\boxed{
q(x_{1:T}\mid x_0)
=
\prod_{t=1}^{T}q(x_t\mid x_{t-1})
}
$$

其中：

$$
\boxed{
q(x_t\mid x_{t-1})
=
\mathcal N
\left(
x_t;
\sqrt{1-\beta_t}x_{t-1},
\beta_tI
\right)
}
\tag{2}
$$

定义：

$$
\alpha_t=1-\beta_t
$$

$$
\bar\alpha_t=\prod_{s=1}^{t}\alpha_s
$$

则：

$$
\boxed{
x_t=
\sqrt{\alpha_t}x_{t-1}
+
\sqrt{\beta_t}\epsilon_t
}
$$

其中：

$$
\epsilon_t\sim\mathcal N(0,I)
$$

各步噪声独立。

### 3.1 推导任意时间步的闭式加噪公式

从：

$$
x_t=
\sqrt{\alpha_t}x_{t-1}
+
\sqrt{\beta_t}\epsilon_t
$$

开始。

假设：

$$
x_{t-1}
=
\sqrt{\bar\alpha_{t-1}}x_0
+
\sqrt{1-\bar\alpha_{t-1}}\epsilon'
$$

代入：

$$
\begin{aligned}
x_t
={}&
\sqrt{\alpha_t\bar\alpha_{t-1}}x_0\\
&+
\sqrt{\alpha_t(1-\bar\alpha_{t-1})}\epsilon'\\
&+
\sqrt{\beta_t}\epsilon_t
\end{aligned}
$$

因为：

$$
\epsilon',\epsilon_t\sim\mathcal N(0,I)
$$

且相互独立，所以两个高斯噪声之和仍为高斯。

其方差：

$$
\begin{aligned}
V_t
&=\alpha_t(1-\bar\alpha_{t-1})+\beta_t\\
&=\alpha_t-\alpha_t\bar\alpha_{t-1}+1-\alpha_t\\
&=1-\bar\alpha_t
\end{aligned}
$$

因此：

$$
\boxed{
x_t=
\sqrt{\bar\alpha_t}x_0
+
\sqrt{1-\bar\alpha_t}\epsilon
}
$$

其中：

$$
\epsilon\sim\mathcal N(0,I)
$$

所以：

$$
\boxed{
q(x_t\mid x_0)
=
\mathcal N
\left(
x_t;
\sqrt{\bar\alpha_t}x_0,
(1-\bar\alpha_t)I
\right)
}
\tag{4}
$$

这个公式说明：

**虽然前向过程是马尔可夫链，但训练时可以直接采样任意时间步的 $x_t$，无需逐步加噪。**

### 3.2 为什么最终能够接近标准高斯？

根据：

$$
x_T=
\sqrt{\bar\alpha_T}x_0
+
\sqrt{1-\bar\alpha_T}\epsilon
$$

如果：

$$
\bar\alpha_T\approx0
$$

则：

$$
\sqrt{\bar\alpha_T}x_0\approx0
$$

且：

$$
\sqrt{1-\bar\alpha_T}\approx1
$$

所以：

$$
\boxed{
x_T\approx\epsilon\sim\mathcal N(0,I)
}
$$

注意，只有在信号衰减足够充分时，才有这个近似。

---

# 第二部分：ELBO 的完整推导

## 四、为什么需要 ELBO？

DDPM 希望最大化：

$$
\log p_\theta(x_0)
$$

但：

$$
p_\theta(x_0)
=
\int p_\theta(x_{0:T})\,dx_{1:T}
$$

这个高维积分通常无法直接计算。

因此，引入固定的前向分布：

$$
q(x_{1:T}\mid x_0)
$$

利用它构造变分下界。

### 4.1 从对数似然出发

$$
\log p_\theta(x_0)
=
\log
\int p_\theta(x_{0:T})\,dx_{1:T}
$$

在积分中乘除：

$$
q(x_{1:T}\mid x_0)
$$

得到：

$$
\log p_\theta(x_0)
=
\log
\int
q(x_{1:T}\mid x_0)
\frac{
p_\theta(x_{0:T})
}{
q(x_{1:T}\mid x_0)
}
dx_{1:T}
$$

写成期望：

$$
\boxed{
\log p_\theta(x_0)
=
\log
\mathbb E_{q(x_{1:T}\mid x_0)}
\left[
\frac{
p_\theta(x_{0:T})
}{
q(x_{1:T}\mid x_0)
}
\right]
}
$$

### 4.2 使用 Jensen 不等式

因为对数函数：

$$
\log x
$$

是凹函数，所以：

$$
\log\mathbb E[X]\geq\mathbb E[\log X]
$$

因此：

$$
\log p_\theta(x_0)
\geq
\mathbb E_q
\left[
\log
\frac{
p_\theta(x_{0:T})
}{
q(x_{1:T}\mid x_0)
}
\right]
$$

定义：

$$
\boxed{
\operatorname{ELBO}(x_0)
=
\mathbb E_q
\left[
\log p_\theta(x_{0:T})
-
\log q(x_{1:T}\mid x_0)
\right]
}
$$

于是：

$$
\boxed{
\log p_\theta(x_0)
\geq
\operatorname{ELBO}(x_0)
}
$$

因此，最大化 ELBO 可以间接提高数据对数似然的下界。

### 4.3 DDPM 为什么最小化负 ELBO？

定义：

$$
\boxed{
L(\theta)
=
-\mathbb E_{x_0\sim q_{\mathrm{data}}}
[\operatorname{ELBO}(x_0)]
}
$$

因此：

$$
\boxed{
\mathbb E[-\log p_\theta(x_0)]
\leq L(\theta)
}
$$

这就是论文公式 (3) 的基本含义：

**DDPM 最小化的是负对数似然的一个变分上界。**

---

## 五、ELBO 与 KL 散度的严格关系

这是理解变分推断非常重要的一步。

根据贝叶斯公式：

$$
p_\theta(x_{1:T}\mid x_0)
=
\frac{
p_\theta(x_{0:T})
}{
p_\theta(x_0)
}
$$

考虑：

$$
D_{\mathrm{KL}}
\left(
q(x_{1:T}\mid x_0)
\|
p_\theta(x_{1:T}\mid x_0)
\right)
$$

根据 KL 定义：

$$
\begin{aligned}
D_{\mathrm{KL}}
={}&
\mathbb E_q
\left[
\log q(x_{1:T}\mid x_0)
-
\log p_\theta(x_{1:T}\mid x_0)
\right]
\end{aligned}
$$

代入贝叶斯公式：

$$
\begin{aligned}
D_{\mathrm{KL}}
={}&
\mathbb E_q
\left[
\log q(x_{1:T}\mid x_0)
-
\log p_\theta(x_{0:T})
+
\log p_\theta(x_0)
\right]
\end{aligned}
$$

由于：

$$
\log p_\theta(x_0)
$$

不依赖潜变量，所以：

$$
D_{\mathrm{KL}}
=
-\operatorname{ELBO}(x_0)
+
\log p_\theta(x_0)
$$

因此：

$$
\boxed{
\log p_\theta(x_0)
=
\operatorname{ELBO}(x_0)
+
D_{\mathrm{KL}}
\left(
q(x_{1:T}\mid x_0)
\|
p_\theta(x_{1:T}\mid x_0)
\right)
}
$$

因为：

$$
D_{\mathrm{KL}}\geq0
$$

所以：

$$
\operatorname{ELBO}(x_0)\leq\log p_\theta(x_0)
$$

这个推导还告诉我们：

**ELBO 与真实对数似然之间的差距，就是前向推断分布与模型真实后验之间的 KL 散度。**

因此，优化 ELBO 实际上同时涉及：

- 提高模型的数据似然。
- 减小推断分布与模型后验之间的差距。

---

## 六、将 DDPM 的 ELBO 分解为多个 KL

现在进入论文最关键的推导。

从负 ELBO 开始：

$$
L
=
\mathbb E_q
\left[
\log
\frac{
q(x_{1:T}\mid x_0)
}{
p_\theta(x_{0:T})
}
\right]
$$

根据前向马尔可夫分解：

$$
q(x_{1:T}\mid x_0)
=
\prod_{t=1}^{T}q(x_t\mid x_{t-1})
$$

反向生成模型：

$$
p_\theta(x_{0:T})
=
p(x_T)
\prod_{t=1}^{T}
p_\theta(x_{t-1}\mid x_t)
$$

因此：

$$
\begin{aligned}
L
=
\mathbb E_q\Bigg[
&\sum_{t=1}^{T}\log q(x_t\mid x_{t-1})\\
&-\log p(x_T)\\
&-\sum_{t=1}^{T}\log p_\theta(x_{t-1}\mid x_t)
\Bigg]
\end{aligned}
$$

但这个形式还不能直接看出各个时间步的 KL。

关键是将前向马尔可夫链改写为另一种条件分解。

### 6.1 前向链的反向条件分解

根据概率链式法则：

$$
q(x_{1:T}\mid x_0)
=
q(x_T\mid x_0)
\prod_{t=2}^{T}
q(x_{t-1}\mid x_t,x_0)
$$

这里的：

$$
q(x_{t-1}\mid x_t,x_0)
$$

是前向扩散过程的后验分布。

注意，它不是生成模型：

$$
p_\theta(x_{t-1}\mid x_t)
$$

两者是不同的条件分布。

代入负 ELBO：

$$
\begin{aligned}
L
=
\mathbb E_q\Bigg[
&\log q(x_T\mid x_0)\\
&+\sum_{t=2}^{T}
\log q(x_{t-1}\mid x_t,x_0)\\
&-\log p(x_T)\\
&-\sum_{t=1}^{T}
\log p_\theta(x_{t-1}\mid x_t)
\Bigg]
\end{aligned}
$$

将对应项组合：

$$
\begin{aligned}
L
=
\mathbb E_q\Bigg[
&\log\frac{q(x_T\mid x_0)}{p(x_T)}\\
&+\sum_{t=2}^{T}
\log
\frac{
q(x_{t-1}\mid x_t,x_0)
}{
p_\theta(x_{t-1}\mid x_t)
}\\
&-\log p_\theta(x_0\mid x_1)
\Bigg]
\end{aligned}
$$

第一项是：

$$
D_{\mathrm{KL}}(q(x_T\mid x_0)\|p(x_T))
$$

中间每一项是：

$$
D_{\mathrm{KL}}
\left(
q(x_{t-1}\mid x_t,x_0)
\|
p_\theta(x_{t-1}\mid x_t)
\right)
$$

最后一项是重建负对数似然。

因此：

$$
\boxed{
\begin{aligned}
L
=
\mathbb E_q\Bigg[
&
\underbrace{
D_{\mathrm{KL}}
\left(
q(x_T\mid x_0)\|p(x_T)
\right)
}_{L_T}\\
&+
\sum_{t=2}^{T}
\underbrace{
D_{\mathrm{KL}}
\left(
q(x_{t-1}\mid x_t,x_0)
\|
p_\theta(x_{t-1}\mid x_t)
\right)
}_{L_{t-1}}\\
&-
\underbrace{
\log p_\theta(x_0\mid x_1)
}_{\text{重建项 }L_0}
\Bigg]
\end{aligned}
}
\tag{5}
$$

这就是论文公式 (5)。

它将复杂的 ELBO 分解成三个部分。

### 6.2 第一部分：终点先验 KL

$$
\boxed{
L_T
=
D_{\mathrm{KL}}
\left(
q(x_T\mid x_0)
\|
p(x_T)
\right)
}
$$

其中：

$$
p(x_T)=\mathcal N(0,I)
$$

而：

$$
q(x_T\mid x_0)
=
\mathcal N
\left(
\sqrt{\bar\alpha_T}x_0,
(1-\bar\alpha_T)I
\right)
$$

如果：

$$
\bar\alpha_T\approx0
$$

则：

$$
q(x_T\mid x_0)\approx\mathcal N(0,I)
$$

因此：

$$
L_T\approx0
$$

更重要的是，在原论文中 $\beta_t$ 固定，因此 $L_T$ 不依赖 $\theta$，训练时可以忽略其梯度。

### 6.3 第二部分：反向转移 KL

$$
\boxed{
L_{t-1}
=
D_{\mathrm{KL}}
\left(
q(x_{t-1}\mid x_t,x_0)
\|
p_\theta(x_{t-1}\mid x_t)
\right)
}
$$

它要求学习的反向转移分布：

$$
p_\theta(x_{t-1}\mid x_t)
$$

尽可能接近前向过程的解析后验：

$$
q(x_{t-1}\mid x_t,x_0)
$$

这是 DDPM 训练的核心。

### 6.4 第三部分：数据重建项

$$
\boxed{
L_0
=
-\log p_\theta(x_0\mid x_1)
}
$$

它衡量从最后一个轻微加噪状态 $x_1$ 恢复原始数据 $x_0$ 的能力。

原论文针对离散图像像素，使用离散化高斯解码器计算这一项，而不是简单地把所有时间步都视为连续高斯密度回归。

---

# 第三部分：前向后验的解析推导

## 七、为什么 $q(x_{t-1}\mid x_t,x_0)$ 是高斯分布？

这是 DDPM 最重要的数学公式之一，也是后面所有噪声预测公式的基础。

先明确三个分布。

**第一个：单步前向转移**

$$
\boxed{
q(x_t\mid x_{t-1})
=
\mathcal N(\sqrt{\alpha_t}x_{t-1},\beta_tI)
}
$$

**第二个：从原始数据直接加噪到 $t-1$**

$$
\boxed{
q(x_{t-1}\mid x_0)
=
\mathcal N
\left(
\sqrt{\bar\alpha_{t-1}}x_0,
(1-\bar\alpha_{t-1})I
\right)
}
$$

**第三个：我们需要计算的后验**

$$
\boxed{
q(x_{t-1}\mid x_t,x_0)
}
$$

为什么额外给定 $x_0$？

因为只知道 $x_t$ 时，之前的状态可能来自多个不同的数据模态，后验通常很复杂。

但是给定 $x_0$ 后，前向链中相关的条件分布都是高斯，可以利用高斯分布乘积的性质得到解析解。

### 7.1 使用贝叶斯公式

根据贝叶斯公式：

$$
q(x_{t-1}\mid x_t,x_0)
=
\frac{
q(x_t\mid x_{t-1},x_0)
q(x_{t-1}\mid x_0)
}{
q(x_t\mid x_0)
}
$$

由于前向过程满足马尔可夫性：

$$
q(x_t\mid x_{t-1},x_0)
=
q(x_t\mid x_{t-1})
$$

因此：

$$
\boxed{
q(x_{t-1}\mid x_t,x_0)
\propto
q(x_t\mid x_{t-1})
q(x_{t-1}\mid x_0)
}
$$

这里使用正比符号，是因为分母：

$$
q(x_t\mid x_0)
$$

对于待求变量 $x_{t-1}$ 来说是常数。

接下来把右侧两个高斯密度相乘。

### 7.2 展开两个高斯分布

为了简化符号，令：

$$
y=x_{t-1}
$$

那么：

$$
q(x_t\mid y)
\propto
\exp
\left[
-\frac{
\|x_t-\sqrt{\alpha_t}y\|^2
}{
2\beta_t
}
\right]
$$

另外：

$$
q(y\mid x_0)
\propto
\exp
\left[
-\frac{
\|y-\sqrt{\bar\alpha_{t-1}}x_0\|^2
}{
2(1-\bar\alpha_{t-1})
}
\right]
$$

因此：

$$
\begin{aligned}
q(y\mid x_t,x_0)
\propto
\exp\Bigg[
&-\frac{
\|x_t-\sqrt{\alpha_t}y\|^2
}{
2\beta_t
}\\
&-\frac{
\|y-\sqrt{\bar\alpha_{t-1}}x_0\|^2
}{
2(1-\bar\alpha_{t-1})
}
\Bigg]
\end{aligned}
$$

现在需要将指数整理成高斯分布的标准形式。

### 7.3 展开平方项

第一个平方：

$$
\begin{aligned}
\|x_t-\sqrt{\alpha_t}y\|^2
={}&
\|x_t\|^2
-2\sqrt{\alpha_t}x_t^\top y
+\alpha_t\|y\|^2
\end{aligned}
$$

第二个平方：

$$
\begin{aligned}
\|y-\sqrt{\bar\alpha_{t-1}}x_0\|^2
={}&
\|y\|^2\\
&-2\sqrt{\bar\alpha_{t-1}}x_0^\top y\\
&+\bar\alpha_{t-1}\|x_0\|^2
\end{aligned}
$$

只保留与 $y$ 有关的部分：

$$
\begin{aligned}
\log q(y\mid x_t,x_0)
={}&
-\frac12
\left[
\frac{\alpha_t}{\beta_t}
+
\frac1{1-\bar\alpha_{t-1}}
\right]\|y\|^2\\
&+
\left[
\frac{\sqrt{\alpha_t}}{\beta_t}x_t
+
\frac{\sqrt{\bar\alpha_{t-1}}}{1-\bar\alpha_{t-1}}x_0
\right]^\top y
+C
\end{aligned}
$$

其中 $C$ 不依赖 $y$。

定义：

$$
\boxed{
A_t=
\frac{\alpha_t}{\beta_t}
+
\frac1{1-\bar\alpha_{t-1}}
}
$$

以及：

$$
\boxed{
b_t=
\frac{\sqrt{\alpha_t}}{\beta_t}x_t
+
\frac{\sqrt{\bar\alpha_{t-1}}}{1-\bar\alpha_{t-1}}x_0
}
$$

于是：

$$
\log q(y\mid x_t,x_0)
=
-\frac12A_t\|y\|^2+b_t^\top y+C
$$

### 7.4 通过配方得到高斯均值和方差

我们知道：

$$
-\frac12A_t\|y\|^2+b_t^\top y
$$

可以写成：

$$
-\frac{A_t}{2}
\left\|
y-\frac{b_t}{A_t}
\right\|^2+C'
$$

因此：

$$
\boxed{
q(y\mid x_t,x_0)
=
\mathcal N
\left(
\frac{b_t}{A_t},
\frac1{A_t}I
\right)
}
$$

这说明：

$$
q(x_{t-1}\mid x_t,x_0)
$$

确实是高斯分布。

接下来分别化简它的方差和均值。

---

## 八、后验方差 $\tilde\beta_t$ 的推导

根据：

$$
\tilde\beta_t=\frac1{A_t}
$$

得到：

$$
\tilde\beta_t
=
\left[
\frac{\alpha_t}{\beta_t}
+
\frac1{1-\bar\alpha_{t-1}}
\right]^{-1}
$$

通分：

$$
\tilde\beta_t
=
\frac{
\beta_t(1-\bar\alpha_{t-1})
}{
\alpha_t(1-\bar\alpha_{t-1})+\beta_t
}
$$

现在化简分母：

$$
\begin{aligned}
&\alpha_t(1-\bar\alpha_{t-1})+\beta_t\\
&=\alpha_t-\alpha_t\bar\alpha_{t-1}+\beta_t\\
&=\alpha_t+\beta_t-\bar\alpha_t
\end{aligned}
$$

因为：

$$
\alpha_t+\beta_t=1
$$

所以：

$$
\alpha_t(1-\bar\alpha_{t-1})+\beta_t
=
1-\bar\alpha_t
$$

因此：

$$
\boxed{
\tilde\beta_t
=
\frac{
1-\bar\alpha_{t-1}
}{
1-\bar\alpha_t
}\beta_t
}
\tag{7}
$$

这就是论文中的后验方差。

---

## 九、后验均值 $\tilde\mu_t$ 的推导

根据：

$$
\tilde\mu_t=\frac{b_t}{A_t}
$$

由于：

$$
\frac1{A_t}=\tilde\beta_t
$$

所以：

$$
\tilde\mu_t
=
\tilde\beta_t
\left[
\frac{\sqrt{\alpha_t}}{\beta_t}x_t
+
\frac{\sqrt{\bar\alpha_{t-1}}}{1-\bar\alpha_{t-1}}x_0
\right]
$$

分别计算两个系数。

### 9.1 $x_t$ 的系数

$$
\tilde\beta_t
\frac{\sqrt{\alpha_t}}{\beta_t}
$$

代入：

$$
\tilde\beta_t
=
\frac{
1-\bar\alpha_{t-1}
}{
1-\bar\alpha_t
}\beta_t
$$

得到：

$$
\boxed{
\frac{
\sqrt{\alpha_t}(1-\bar\alpha_{t-1})
}{
1-\bar\alpha_t
}
}
$$

### 9.2 $x_0$ 的系数

$$
\tilde\beta_t
\frac{\sqrt{\bar\alpha_{t-1}}}{1-\bar\alpha_{t-1}}
$$

代入：

$$
\boxed{
\frac{
\sqrt{\bar\alpha_{t-1}}\beta_t
}{
1-\bar\alpha_t
}
}
$$

因此：

$$
\boxed{
\begin{aligned}
\tilde\mu_t(x_t,x_0)
={}&
\frac{
\sqrt{\bar\alpha_{t-1}}\beta_t
}{
1-\bar\alpha_t
}x_0\\
&+
\frac{
\sqrt{\alpha_t}(1-\bar\alpha_{t-1})
}{
1-\bar\alpha_t
}x_t
\end{aligned}
}
\tag{7}
$$

最终：

$$
\boxed{
q(x_{t-1}\mid x_t,x_0)
=
\mathcal N
\left(
\tilde\mu_t(x_t,x_0),
\tilde\beta_tI
\right)
}
\tag{6}
$$

这就是论文公式 (6)、(7) 的完整来源。

**需要特别理解：这个高斯后验是由贝叶斯公式和两个高斯分布相乘严格推导得到的，而不是神经网络学习出来的。**

---

# 第四部分：从 KL 推导噪声预测 MSE

## 十、为什么高斯 KL 可以转化为均值 MSE？

现在我们已经知道：

$$
q(x_{t-1}\mid x_t,x_0)
=
\mathcal N(\tilde\mu_t,\tilde\beta_tI)
$$

而生成模型定义：

$$
p_\theta(x_{t-1}\mid x_t)
=
\mathcal N(\mu_\theta,\sigma_t^2I)
$$

其中：

- $\tilde\mu_t$：真实前向后验均值。
- $\tilde\beta_t$：真实前向后验方差。
- $\mu_\theta$：神经网络参数化的反向均值。
- $\sigma_t^2$：模型反向方差，原论文采用固定值。

我们的目标是：

$$
\boxed{
L_{t-1}
=
D_{\mathrm{KL}}
\left(
\mathcal N(\tilde\mu_t,\tilde\beta_tI)
\|
\mathcal N(\mu_\theta,\sigma_t^2I)
\right)
}
$$

### 10.1 多维高斯 KL 的一般公式

对于：

$$
P=\mathcal N(\mu_P,\Sigma_P)
$$

$$
Q=\mathcal N(\mu_Q,\Sigma_Q)
$$

有：

$$
\boxed{
\begin{aligned}
D_{\mathrm{KL}}(P\|Q)
=
\frac12\Big[
&\operatorname{tr}(\Sigma_Q^{-1}\Sigma_P)\\
&+(\mu_Q-\mu_P)^\top
\Sigma_Q^{-1}(\mu_Q-\mu_P)\\
&-D
+\log\frac{\det\Sigma_Q}{\det\Sigma_P}
\Big]
\end{aligned}
}
$$

其中：

$$
D
$$

是数据维度。

### 10.2 代入 DDPM 的两个高斯分布

因为：

$$
\Sigma_P=\tilde\beta_tI
$$

$$
\Sigma_Q=\sigma_t^2I
$$

所以：

$$
\Sigma_Q^{-1}=\frac1{\sigma_t^2}I
$$

第一项：

$$
\operatorname{tr}(\Sigma_Q^{-1}\Sigma_P)
=
D\frac{\tilde\beta_t}{\sigma_t^2}
$$

第二项：

$$
(\mu_\theta-\tilde\mu_t)^\top
\Sigma_Q^{-1}
(\mu_\theta-\tilde\mu_t)
=
\frac{
\|\mu_\theta-\tilde\mu_t\|^2
}{
\sigma_t^2
}
$$

行列式项：

$$
\log
\frac{\det(\sigma_t^2I)}{\det(\tilde\beta_tI)}
=
D\log\frac{\sigma_t^2}{\tilde\beta_t}
$$

因此：

$$
\boxed{
\begin{aligned}
L_{t-1}
=
\frac12\Bigg[
&D\frac{\tilde\beta_t}{\sigma_t^2}
+
\frac{
\|\mu_\theta-\tilde\mu_t\|^2
}{
\sigma_t^2
}\\
&-D
+
D\log\frac{\sigma_t^2}{\tilde\beta_t}
\Bigg]
\end{aligned}
}
$$

由于原论文固定反向方差：

$$
\sigma_t^2
$$

所以除了均值误差以外，其余项都不依赖 $\theta$。

记为常数 $C_t$，得到：

$$
\boxed{
L_{t-1}
=
\mathbb E_q
\left[
\frac1{2\sigma_t^2}
\|\tilde\mu_t(x_t,x_0)-\mu_\theta(x_t,t)\|^2
\right]
+C_t
}
\tag{8}
$$

这就是论文公式 (8)。

因此：

**在固定反向方差的条件下，最小化每一步反向转移的 KL，等价于最小化反向高斯均值的加权平方误差。**

---

## 十一、为什么不直接预测后验均值？

既然：

$$
L_{t-1}
\propto
\|\tilde\mu_t-\mu_\theta\|^2
$$

最直接的方法就是让网络预测：

$$
\mu_\theta(x_t,t)
$$

并监督：

$$
\tilde\mu_t(x_t,x_0)
$$

这在数学上完全可行。

但是论文进一步发现，可以利用前向加噪公式，将后验均值重新表示为噪声 $\epsilon$ 的函数。

这样网络就可以预测：

$$
\epsilon_\theta(x_t,t)
$$

而不必直接预测：

$$
\mu_\theta(x_t,t)
$$

下面推导这个关系。

---

## 十二、从后验均值推导噪声预测参数化

我们已经得到：

$$
\tilde\mu_t
=
\frac{
\sqrt{\bar\alpha_{t-1}}\beta_t
}{
1-\bar\alpha_t
}x_0
+
\frac{
\sqrt{\alpha_t}(1-\bar\alpha_{t-1})
}{
1-\bar\alpha_t
}x_t
$$

另一方面，前向加噪公式：

$$
\boxed{
x_t=
\sqrt{\bar\alpha_t}x_0
+
\sqrt{1-\bar\alpha_t}\epsilon
}
$$

可以解出：

$$
\boxed{
x_0=
\frac{
x_t-\sqrt{1-\bar\alpha_t}\epsilon
}{
\sqrt{\bar\alpha_t}
}
}
$$

将它代入后验均值。

### 12.1 代入 $x_0$

$$
\begin{aligned}
\tilde\mu_t
={}&
\frac{
\sqrt{\bar\alpha_{t-1}}\beta_t
}{
1-\bar\alpha_t
}
\frac{
x_t-\sqrt{1-\bar\alpha_t}\epsilon
}{
\sqrt{\bar\alpha_t}
}\\
&+
\frac{
\sqrt{\alpha_t}(1-\bar\alpha_{t-1})
}{
1-\bar\alpha_t
}x_t
\end{aligned}
$$

因为：

$$
\bar\alpha_t=\alpha_t\bar\alpha_{t-1}
$$

所以：

$$
\frac{
\sqrt{\bar\alpha_{t-1}}
}{
\sqrt{\bar\alpha_t}
}
=
\frac1{\sqrt{\alpha_t}}
$$

因此：

$$
\begin{aligned}
\tilde\mu_t
={}&
\frac{\beta_t}{
\sqrt{\alpha_t}(1-\bar\alpha_t)
}x_t\\
&-
\frac{\beta_t}{
\sqrt{\alpha_t}\sqrt{1-\bar\alpha_t}
}\epsilon\\
&+
\frac{
\sqrt{\alpha_t}(1-\bar\alpha_{t-1})
}{
1-\bar\alpha_t
}x_t
\end{aligned}
$$

### 12.2 合并 $x_t$ 的系数

两个 $x_t$ 系数相加：

$$
\frac{\beta_t}{
\sqrt{\alpha_t}(1-\bar\alpha_t)
}
+
\frac{
\sqrt{\alpha_t}(1-\bar\alpha_{t-1})
}{
1-\bar\alpha_t
}
$$

通分：

$$
=
\frac{
\beta_t+\alpha_t(1-\bar\alpha_{t-1})
}{
\sqrt{\alpha_t}(1-\bar\alpha_t)
}
$$

分子：

$$
\begin{aligned}
\beta_t+\alpha_t(1-\bar\alpha_{t-1})
&=\beta_t+\alpha_t-\bar\alpha_t\\
&=1-\bar\alpha_t
\end{aligned}
$$

所以：

$$
\boxed{
\frac{
\beta_t+\alpha_t(1-\bar\alpha_{t-1})
}{
\sqrt{\alpha_t}(1-\bar\alpha_t)
}
=
\frac1{\sqrt{\alpha_t}}
}
$$

最终：

$$
\boxed{
\tilde\mu_t(x_t,x_0)
=
\frac1{\sqrt{\alpha_t}}
\left(
x_t-
\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\epsilon
\right)
}
$$

这个公式非常重要。

它说明：

**给定 $x_t$ 后，只要知道从 $x_0$ 加噪到 $x_t$ 时使用的总噪声 $\epsilon$，就能够计算前向后验均值。**

因此，我们可以让神经网络预测：

$$
\epsilon_\theta(x_t,t)
$$

并定义：

$$
\boxed{
\mu_\theta(x_t,t)
=
\frac1{\sqrt{\alpha_t}}
\left(
x_t-
\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}
\epsilon_\theta(x_t,t)
\right)
}
\tag{11}
$$

这就是 DDPM 最经典的噪声预测参数化。

---

## 十三、将噪声预测参数化代回 KL

前面得到：

$$
L_{t-1}
=
\mathbb E
\left[
\frac1{2\sigma_t^2}
\|\tilde\mu_t-\mu_\theta\|^2
\right]+C_t
$$

现在：

$$
\tilde\mu_t
=
\frac1{\sqrt{\alpha_t}}
\left(
x_t-
\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\epsilon
\right)
$$

而：

$$
\mu_\theta
=
\frac1{\sqrt{\alpha_t}}
\left(
x_t-
\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}
\epsilon_\theta
\right)
$$

两者相减：

$$
\begin{aligned}
\tilde\mu_t-\mu_\theta
&=
\frac1{\sqrt{\alpha_t}}
\left[
-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\epsilon
+
\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\epsilon_\theta
\right]\\
&=
\frac{\beta_t}{
\sqrt{\alpha_t}\sqrt{1-\bar\alpha_t}
}
(\epsilon_\theta-\epsilon)
\end{aligned}
$$

所以：

$$
\boxed{
\|\tilde\mu_t-\mu_\theta\|^2
=
\frac{\beta_t^2}{
\alpha_t(1-\bar\alpha_t)
}
\|\epsilon-\epsilon_\theta\|^2
}
$$

代回：

$$
\boxed{
L_{t-1}
=
\mathbb E_{x_0,\epsilon}
\left[
\frac{
\beta_t^2
}{
2\sigma_t^2\alpha_t(1-\bar\alpha_t)
}
\left\|
\epsilon-
\epsilon_\theta(
\sqrt{\bar\alpha_t}x_0+
\sqrt{1-\bar\alpha_t}\epsilon,t
)
\right\|^2
\right]
+C_t
}
\tag{12}
$$

这就是论文公式 (12)。

定义时间步权重：

$$
\boxed{
w_t=
\frac{
\beta_t^2
}{
2\sigma_t^2\alpha_t(1-\bar\alpha_t)
}
}
$$

那么：

$$
\boxed{
L_{t-1}
=
w_t
\mathbb E
\|\epsilon-\epsilon_\theta(x_t,t)\|^2
+C_t
}
$$

至此，我们完成了整条核心推导：

$$
\boxed{
\begin{aligned}
&\text{ELBO}\\
&\Downarrow\\
&\text{反向转移 KL}\\
&\Downarrow\\
&\text{高斯均值 MSE}\\
&\Downarrow\\
&\text{后验均值的噪声参数化}\\
&\Downarrow\\
&\text{加权噪声预测 MSE}
\end{aligned}
}
$$

**所以 DDPM 的噪声预测 MSE 有严格的变分推断来源，并不是单纯把去噪任务当作监督学习。**

但这里还有最后一个重要问题：论文最终为什么把权重 $w_t$ 去掉？

---

# 第五部分：为什么最终使用简化损失 $L_{\mathrm{simple}}$？

## 十四、原始 ELBO 与简化训练目标的区别

从前面的推导可知，严格的反向 KL 对应：

$$
L_{t-1}
=
w_t
\mathbb E
\|\epsilon-\epsilon_\theta(x_t,t)\|^2+C_t
$$

其中：

$$
w_t=
\frac{
\beta_t^2
}{
2\sigma_t^2\alpha_t(1-\bar\alpha_t)
}
$$

如果严格优化原始 ELBO，就需要保留这些时间步权重，并正确处理 $L_0$ 和 $L_T$。

但是论文发现，去掉这些权重之后，生成样本质量反而更好。

因此定义：

$$
\boxed{
L_{\mathrm{simple}}(\theta)
=
\mathbb E_{t,x_0,\epsilon}
\left[
\left\|
\epsilon-
\epsilon_\theta(
\sqrt{\bar\alpha_t}x_0+
\sqrt{1-\bar\alpha_t}\epsilon,t
)
\right\|^2
\right]
}
\tag{14}
$$

其中：

$$
t\sim\operatorname{Uniform}\{1,\ldots,T\}
$$

$$
x_0\sim q_{\mathrm{data}}
$$

$$
\epsilon\sim\mathcal N(0,I)
$$

这个损失非常简单：

1. 随机采样真实数据 $x_0$。
2. 随机采样时间步 $t$。
3. 随机采样高斯噪声 $\epsilon$。
4. 直接构造 $x_t$。
5. 让网络预测添加的噪声。
6. 最小化噪声预测 MSE。

### 14.1 去掉权重是否仍然等价于 ELBO？

严格来说：

$$
\boxed{
L_{\mathrm{simple}}\neq L_{\mathrm{ELBO}}
}
$$

因为不同时间步的权重已经发生改变。

对于固定的 $t$，只要权重为正，单独最小化该时间步的 MSE，不会改变其函数空间中的最优预测：

$$
\epsilon_\theta^*(x_t,t)
=
\mathbb E[\epsilon\mid x_t]
$$

但是所有时间步共享同一个网络：

$$
\epsilon_\theta(x_t,t)
$$

网络容量有限时，不同权重会影响各个时间步的优化程度。

因此：

**去掉权重不会改变每个时间步理想情况下的最优预测目标，但可能改变有限容量网络的实际训练结果。**

### 14.2 为什么去掉权重能够改善生成质量？

原论文认为，简化目标重新分配了不同时间步的训练权重。

尤其是在论文采用的噪声调度下，简化损失相对于原始变分目标，降低了部分小噪声时间步的相对权重。

小噪声时间步主要处理：

- 局部细节。
- 轻微噪声。
- 接近原始数据的重建。

较大噪声时间步则需要模型学习：

- 更复杂的去噪映射。
- 更大范围的数据结构。
- 从严重退化的数据中恢复有意义的信息。

作者发现，让网络更加关注这些较困难的去噪任务，有助于改善最终生成质量。

不过这属于论文的经验观察，不应理解为所有扩散模型都必须采用均匀时间步权重。

### 14.3 论文实验验证

在 CIFAR-10 上，论文报告：

| 训练目标 | FID | Inception Score |
|---|---:|---:|
| 原始变分目标 $L$，固定各向同性方差 | 13.51 | $7.67\pm0.13$ |
| 简化目标 $L_{\mathrm{simple}}$ | **3.17** | **$9.46\pm0.11$** |

同时，使用简化目标的模型在负对数似然指标上并没有表现得更好。

这说明：

$$
\boxed{
\text{更好的似然指标}
\not\Rightarrow
\text{更好的感知生成质量}
}
$$

这也是 DDPM 论文非常重要的实验结论。[arXiv](https://arxiv.org/pdf/2006.11239?utm_source=chatgpt.com)

---

# 第六部分：DDPM 与 Score Matching 的数学联系

## 十五、噪声预测为什么等价于 Score 预测？

我们刚刚学习了 NCSN，因此这里必须把两者联系起来。

DDPM 的前向加噪公式：

$$
\boxed{
x_t=
\sqrt{\bar\alpha_t}x_0+
\sqrt{1-\bar\alpha_t}\epsilon
}
$$

定义：

$$
a_t=\sqrt{\bar\alpha_t}
$$

$$
b_t=\sqrt{1-\bar\alpha_t}
$$

那么：

$$
x_t=a_tx_0+b_t\epsilon
$$

给定 $x_0$ 时：

$$
q(x_t\mid x_0)
=
\mathcal N(a_tx_0,b_t^2I)
$$

因此条件 Score：

$$
\boxed{
\nabla_{x_t}\log q(x_t\mid x_0)
=
-\frac{x_t-a_tx_0}{b_t^2}
}
$$

因为：

$$
x_t-a_tx_0=b_t\epsilon
$$

所以：

$$
\boxed{
\nabla_{x_t}\log q(x_t\mid x_0)
=
-\frac{\epsilon}{b_t}
}
$$

也就是：

$$
\boxed{
\nabla_{x_t}\log q(x_t\mid x_0)
=
-\frac{\epsilon}{\sqrt{1-\bar\alpha_t}}
}
$$

但我们真正希望学习的是边缘 Score：

$$
\nabla_{x_t}\log q_t(x_t)
$$

根据之前 NCSN 中推导的 DSM 恒等式：

$$
\boxed{
\nabla_{x_t}\log q_t(x_t)
=
\mathbb E
\left[
-\frac{\epsilon}{\sqrt{1-\bar\alpha_t}}
\;\middle|\;
x_t
\right]
}
$$

而 MSE 的最优预测满足：

$$
\boxed{
\epsilon_\theta^*(x_t,t)
=
\mathbb E[\epsilon\mid x_t]
}
$$

所以：

$$
\boxed{
\nabla_{x_t}\log q_t(x_t)
=
-\frac{
\epsilon_\theta^*(x_t,t)
}{
\sqrt{1-\bar\alpha_t}
}
}
$$

因此可以定义 Score 网络：

$$
\boxed{
s_\theta(x_t,t)
=
-\frac{
\epsilon_\theta(x_t,t)
}{
\sqrt{1-\bar\alpha_t}
}
}
$$

这就是 DDPM 与 NCSN 在训练目标上的数学联系。

### 15.1 为什么 DDPM 和 NCSN 的加噪形式不同？

NCSN 使用：

$$
x=x_0+\sigma\epsilon
$$

DDPM 使用：

$$
x_t=
\sqrt{\bar\alpha_t}x_0+
\sqrt{1-\bar\alpha_t}\epsilon
$$

区别在于：

**NCSN 的加噪过程不显式缩放原始信号，而 DDPM 同时衰减原始信号并添加噪声。**

不过，对固定的 $t$，如果令：

$$
y_t=\frac{x_t}{\sqrt{\bar\alpha_t}}
$$

则：

$$
y_t
=
x_0+
\sqrt{
\frac{1-\bar\alpha_t}{\bar\alpha_t}
}\epsilon
$$

因此，在：

$$
\bar\alpha_t>0
$$

时，DDPM 的边缘分布也可以通过变量缩放，转化为类似 NCSN 的加性高斯扰动形式。

其等效噪声标准差为：

$$
\boxed{
\sigma_{\mathrm{eff}}(t)
=
\sqrt{
\frac{1-\bar\alpha_t}{\bar\alpha_t}
}
}
$$

这说明两者的训练分布存在直接联系。

但采样动力学仍然不同。

---

## 十六、DDPM 反向更新为什么类似 Langevin Dynamics？

DDPM 反向均值：

$$
\mu_\theta(x_t,t)
=
\frac1{\sqrt{\alpha_t}}
\left(
x_t-
\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}
\epsilon_\theta(x_t,t)
\right)
$$

根据：

$$
s_\theta(x_t,t)
=
-\frac{\epsilon_\theta(x_t,t)}{\sqrt{1-\bar\alpha_t}}
$$

代入：

$$
\boxed{
\mu_\theta(x_t,t)
=
\frac1{\sqrt{\alpha_t}}
\left(
x_t+\beta_ts_\theta(x_t,t)
\right)
}
$$

因此采样：

$$
\boxed{
x_{t-1}
=
\frac1{\sqrt{\alpha_t}}
\left(
x_t+\beta_ts_\theta(x_t,t)
\right)
+
\sigma_tz
}
$$

其中：

$$
z\sim\mathcal N(0,I)
$$

对小 $\beta_t$：

$$
\alpha_t=1-\beta_t
$$

使用一阶泰勒展开：

$$
\frac1{\sqrt{1-\beta_t}}
\approx
1+\frac{\beta_t}{2}
$$

于是：

$$
\begin{aligned}
x_{t-1}
&\approx
\left(1+\frac{\beta_t}{2}\right)
\left(x_t+\beta_ts_\theta(x_t,t)\right)
+\sigma_tz\\
&\approx
x_t+
\frac{\beta_t}{2}x_t+
\beta_ts_\theta(x_t,t)
+\sigma_tz
\end{aligned}
$$

忽略二阶小量后：

$$
\boxed{
x_{t-1}
\approx
x_t+
\beta_t
\left(
s_\theta(x_t,t)+\frac{x_t}{2}
\right)
+
\sigma_tz
}
$$

如果：

$$
\sigma_t^2\approx\beta_t
$$

那么：

$$
\boxed{
x_{t-1}
\approx
x_t+
\beta_t
\left(
s_\theta(x_t,t)+\frac{x_t}{2}
\right)
+
\sqrt{\beta_t}z
}
$$

这个公式与 Langevin Dynamics 非常相似：

$$
x_{k+1}
=
x_k+
\frac{\eta}{2}s(x_k)
+
\sqrt{\eta}z_k
$$

两者都包含：

- Score 引导项。
- 高斯随机扰动。

但 DDPM 还包含：

$$
\frac{\beta_t}{2}x_t
$$

这一项，并且 Score 随扩散时间 $t$ 变化。

因此，**DDPM 反向采样是类似 Langevin 的时间非齐次随机过程，而不是直接在固定分布上运行普通 Langevin Dynamics。**

---

# 第七部分：DDPM 的完整训练与采样算法

## 十七、Algorithm 1：训练过程

原论文的训练算法非常简洁，但每一步都有前面的概率推导作为依据。

### 17.1 训练输入与输出

输入：

$$
x_t,\quad t
$$

输出：

$$
\boxed{
\epsilon_\theta(x_t,t)\in\mathbb R^D
}
$$

网络输出与输入数据具有相同维度。

对于图像：

$$
x_t\in\mathbb R^{C\times H\times W}
$$

则：

$$
\epsilon_\theta(x_t,t)\in\mathbb R^{C\times H\times W}
$$

训练时执行：

$$
x_0\sim q_{\mathrm{data}}
$$

$$
t\sim\operatorname{Uniform}\{1,\ldots,T\}
$$

$$
\epsilon\sim\mathcal N(0,I)
$$

然后：

$$
\boxed{
x_t=
\sqrt{\bar\alpha_t}x_0+
\sqrt{1-\bar\alpha_t}\epsilon
}
$$

最后计算：

$$
\boxed{
L_{\mathrm{simple}}
=
\|\epsilon-\epsilon_\theta(x_t,t)\|^2
}
$$

使用梯度下降更新模型参数。

整个训练过程中：

**不需要运行反向采样链，也不需要依次执行全部 $T$ 个前向扩散步骤。**

---

## 十八、Algorithm 2：反向采样

训练完成后，首先采样：

$$
\boxed{
x_T\sim\mathcal N(0,I)
}
$$

然后从：

$$
t=T
$$

逐渐下降到：

$$
t=1
$$

每一步预测：

$$
\epsilon_\theta(x_t,t)
$$

并计算反向均值：

$$
\boxed{
\mu_\theta(x_t,t)
=
\frac1{\sqrt{\alpha_t}}
\left(
x_t-
\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}
\epsilon_\theta(x_t,t)
\right)
}
$$

随后从反向高斯分布采样：

$$
\boxed{
x_{t-1}
=
\mu_\theta(x_t,t)+\sigma_tz
}
$$

其中：

$$
z\sim\mathcal N(0,I)
$$

对于最后一步：

$$
t=1
$$

原论文设置：

$$
z=0
$$

也就是不再添加随机噪声。

最终返回：

$$
x_0
$$

因此完整采样公式为：

$$
\boxed{
x_{t-1}
=
\frac1{\sqrt{\alpha_t}}
\left(
x_t-
\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}
\epsilon_\theta(x_t,t)
\right)
+
\sigma_tz
}
$$

这就是 DDPM 最经典的反向采样公式。

### 18.1 反向方差应该怎么选择？

原论文没有让网络学习反向方差，而是使用固定值。

主要比较：

$$
\boxed{
\sigma_t^2=\beta_t
}
$$

以及：

$$
\boxed{
\sigma_t^2=\tilde\beta_t
}
$$

其中：

$$
\tilde\beta_t
=
\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\beta_t
$$

实验发现，两种选择的结果比较接近。

需要注意：

$$
\tilde\beta_1=0
$$

因此 $t=1$ 的最后一步不能直接照搬普通连续高斯 KL 的公式。

原论文专门使用离散化高斯解码器处理 $L_0$，采样时最后一步直接输出均值。

### 18.2 为什么反向采样仍然需要随机噪声？

如果直接使用：

$$
x_{t-1}=\mu_\theta(x_t,t)
$$

那么每一步都是确定性的均值更新。

但 DDPM 的生成模型定义为：

$$
p_\theta(x_{t-1}\mid x_t)
=
\mathcal N(\mu_\theta,\sigma_t^2I)
$$

因此，要真正从这个反向转移分布采样，就需要：

$$
\sigma_tz
$$

随机项。

这与 NCSN 中 Langevin Dynamics 的随机扰动具有相似的作用，但二者的概率过程并不完全相同。

---

# 第八部分：原论文的其他重要内容

## 十九、为什么 DDPM 的最后一步需要特殊处理？

对应论文第 3.3 节。

真实图像通常是离散的 8-bit 像素数据：

$$
x_0^{\mathrm{raw}}\in\{0,1,\ldots,255\}^D
$$

论文将其缩放到：

$$
[-1,1]^D
$$

但这并没有改变原始图像取值是离散的事实。

如果直接使用连续高斯密度：

$$
\mathcal N(x_0;\mu_\theta,\sigma_1^2I)
$$

计算离散图像的概率，会涉及概率密度与离散概率质量之间的区别。

因此，作者使用离散化高斯解码器。

对于某个像素值 $x_0^i$，其概率由对应量化区间内的高斯积分给出：

$$
\boxed{
p_\theta(x_0^i\mid x_1)
=
\int_{\delta_-(x_0^i)}^{\delta_+(x_0^i)}
\mathcal N
\left(
u;\mu_\theta^i(x_1,1),\sigma_1^2
\right)du
}
$$

对于非边界像素：

$$
\delta_-(x)=x-\frac1{255}
$$

$$
\delta_+(x)=x+\frac1{255}
$$

边界像素的积分范围向无穷延伸，以正确计算最外侧量化区间的概率质量。

各个像素维度的条件概率相乘：

$$
\boxed{
p_\theta(x_0\mid x_1)
=
\prod_{i=1}^D
p_\theta(x_0^i\mid x_1)
}
$$

这使模型能够为离散图像定义有效的概率质量函数，从而计算以 bits/dim 表示的负对数似然上界。

这一设计主要服务于严格的似然评估，而常见的 DDPM 教学代码通常只实现简化噪声 MSE，不会完整实现离散化高斯解码器。

---

## 二十、论文为什么讨论渐进式压缩？

对应论文第 4.3 节。

这部分经常被忽略，但实际上是论文的另一个重要理论观点。

前面已经得到：

$$
L=L_T+\sum_{t=1}^{T-1}L_t+L_0
$$

其中：

- $L_T$：终点先验匹配。
- $L_t$：各个反向转移的 KL。
- $L_0$：数据重建项。

作者从信息论角度解释这些项。

可以将：

$$
\boxed{
R=L_T+\sum_{t=1}^{T-1}L_t
}
$$

理解为与编码潜变量相关的 Rate。

将：

$$
\boxed{
D=L_0
}
$$

理解为重建失真相关的 Distortion。

因此：

$$
\boxed{
L=R+D
}
$$

这与率失真理论中的思想具有联系。

### 20.1 为什么反向扩散可以理解为渐进式解码？

考虑生成过程：

$$
x_T\rightarrow x_{T-1}\rightarrow\cdots\rightarrow x_0
$$

在较大的 $t$：

$$
x_t
$$

只包含少量原始数据的信息。

随着 $t$ 减小，状态逐渐包含更多信息。

我们可以根据当前 $x_t$ 预测干净数据：

$$
\boxed{
\hat x_0(x_t,t)
=
\frac{
x_t-\sqrt{1-\bar\alpha_t}\epsilon_\theta(x_t,t)
}{
\sqrt{\bar\alpha_t}
}
}
\tag{15}
$$

在采样过程中，不断计算：

$$
\hat x_0(x_T,T),\,
\hat x_0(x_{T-1},T-1),\,
\ldots
$$

就可以观察到：

- 最初主要恢复整体结构。
- 随后逐渐恢复局部细节。
- 最后得到完整图像。

论文认为，这与渐进式有损解码具有相似的性质。

### 20.2 为什么高质量生成不一定意味着最优似然？

论文观察到：

使用严格 ELBO 训练的模型，可以获得更好的编码长度。

但是使用简化目标：

$$
L_{\mathrm{simple}}
$$

训练的模型，生成图像质量反而更好。

作者进一步分析发现，损失编码长度中有相当一部分用于描述人眼不容易察觉的图像细节。

因此：

**似然优化可能花费大量建模能力去拟合对视觉感知不那么重要的细节，而这些细节不一定能显著改善 FID。**

这也是论文从率失真角度解释 DDPM 生成质量的重要动机。

---

## 二十一、DDPM 与自回归模型的联系

论文第 4.3 节还讨论了 DDPM 与自回归解码之间的关系。

自回归模型通常将数据分解为：

$$
p(x_1,\ldots,x_D)
=
\prod_{i=1}^{D}
p(x_i\mid x_{<i})
$$

即按照某个固定顺序逐个生成数据维度。

而 DDPM 使用：

$$
p_\theta(x_{0:T})
=
p(x_T)
\prod_{t=1}^{T}
p_\theta(x_{t-1}\mid x_t)
$$

逐步生成整个数据状态。

作者指出，如果人为设计一种特殊的前向扩散过程：

每一步都遮挡一个新的数据维度，那么反向生成过程就可以退化为类似自回归模型的逐维恢复。

因此，扩散模型可以理解为一种更一般化的渐进式解码方式。

不过这里要注意：

**这并不意味着标准高斯 DDPM 本身就是传统意义上的逐 token 自回归模型。**

它们的条件分解结构、生成变量以及训练机制都不同。

---

# 第九部分：实验与网络结构

## 二十二、原论文的网络结构

DDPM 使用基于 U-Net 的神经网络。

其主要结构包括：

- 卷积残差块。
- 多尺度下采样和上采样。
- Skip Connection。
- Group Normalization。
- Self-Attention。
- 时间步位置编码。

网络输入：

$$
x_t,t
$$

输出：

$$
\epsilon_\theta(x_t,t)
$$

### 22.1 时间步如何输入网络？

作者使用类似 Transformer 的正弦位置编码：

$$
\operatorname{PE}(t)
$$

将时间步编码成向量，再注入不同分辨率的残差块。

因此，网络能够根据：

$$
t
$$

判断当前输入的噪声程度。

与 NCSN 不同，DDPM 不是主要通过噪声条件归一化参数注入时间信息，而是在多个残差模块中使用时间嵌入。

### 22.2 为什么需要 Self-Attention？

卷积擅长提取局部空间结构。

但是图像中的不同位置可能存在长距离依赖。

例如：

- 人脸左右两侧的结构关系。
- 物体不同部分之间的关联。
- 场景中远距离区域的语义一致性。

因此，论文在：

$$
16\times16
$$

分辨率的特征图上使用 Self-Attention。

这有助于模型在去噪过程中建模全局依赖。

---

## 二十三、训练超参数

论文主要实验设置如下：

| 参数 | 设置 |
|---|---|
| 扩散步数 | $T=1000$ |
| 噪声调度 | Linear Schedule |
| $\beta_1$ | $10^{-4}$ |
| $\beta_T$ | $0.02$ |
| 优化器 | Adam |
| CIFAR-10 学习率 | $2\times10^{-4}$ |
| CIFAR-10 Batch Size | 128 |
| 256×256 图像 Batch Size | 64 |
| EMA Decay | 0.9999 |
| 时间编码 | Sinusoidal Embedding |
| 主干网络 | U-Net |
| 反向方差 | 固定 |

这些设置来自论文第 4 节及附录 B。[arXiv](https://arxiv.org/pdf/2006.11239?utm_source=chatgpt.com)

---

## 二十四、主要实验结果

### 24.1 CIFAR-10

论文报告：

$$
\boxed{
\mathrm{FID}=3.17
}
$$

$$
\boxed{
\mathrm{IS}=9.46\pm0.11
}
$$

该结果对应使用：

$$
L_{\mathrm{simple}}
$$

训练的模型。

### 24.2 LSUN

论文还在 LSUN 256×256 数据集上进行了实验。

报告的 FID 包括：

| 数据集 | FID |
|---|---:|
| LSUN Bedroom | 4.90（大模型） |
| LSUN Church | 7.89 |
| LSUN Cat | 19.75 |

这些实验说明，DDPM 不仅能够处理 CIFAR-10 这样的小分辨率图像，也能够扩展到更高分辨率的图像生成。[arXiv](https://arxiv.org/pdf/2006.11239?utm_source=chatgpt.com)

---

# 第十部分：整篇论文的关键问题与总结

## 二十五、几个特别容易理解错误的地方

### 25.1 DDPM 学习的是 $q(x_{t-1}\mid x_t,x_0)$ 吗？

严格来说，不是直接学习这个带有 $x_0$ 条件的后验分布。

训练目标中使用：

$$
q(x_{t-1}\mid x_t,x_0)
$$

作为可以解析计算的监督分布。

但模型输入只有：

$$
x_t,t
$$

没有真实：

$$
x_0
$$

因此，模型实际学习的是：

$$
\boxed{
p_\theta(x_{t-1}\mid x_t)
}
$$

在固定方差、均值 MSE 的条件下，最优均值为：

$$
\boxed{
\mu_\theta^*(x_t,t)
=
\mathbb E[
\tilde\mu_t(x_t,X_0)
\mid x_t
]
}
$$

根据全期望公式：

$$
\mathbb E[
\tilde\mu_t(x_t,X_0)
\mid x_t
]
=
\mathbb E[X_{t-1}\mid x_t]
$$

因此：

$$
\boxed{
\mu_\theta^*(x_t,t)
=
\mathbb E[X_{t-1}\mid x_t]
}
$$

这与之前 DSM 中条件 Score 和边缘 Score 的关系非常相似。

**网络并不是知道真实 $x_0$ 后再计算后验均值，而是在大量训练样本中学习给定 $x_t$ 时的最优条件预测。**

### 25.2 为什么反向过程不一定严格是高斯？

我们严格证明的是：

$$
\boxed{
q(x_{t-1}\mid x_t,x_0)
}
$$

是高斯分布。

但真实反向条件分布：

$$
q(x_{t-1}\mid x_t)
$$

需要对未知 $x_0$ 积分：

$$
\boxed{
q(x_{t-1}\mid x_t)
=
\int
q(x_{t-1}\mid x_t,x_0)
q(x_0\mid x_t)\,dx_0
}
$$

因此，它是一个高斯混合形式的分布，通常不严格是单个高斯。

DDPM 使用单个高斯：

$$
p_\theta(x_{t-1}\mid x_t)
$$

进行近似。

当扩散步长足够小，并满足相应正则性条件时，这种高斯近似具有合理性。

但对于较大的离散时间跨度，真实反向条件分布可能明显偏离单高斯。

这也是后续扩散采样方法不断发展的重要原因之一。

### 25.3 DDPM 的噪声预测是不是在预测实际加入的那次噪声？

训练监督确实是：

$$
\epsilon
$$

但由于网络输入只有：

$$
x_t,t
$$

而不知道真实 $x_0$，所以在 MSE 意义下，理想预测为：

$$
\boxed{
\epsilon_\theta^*(x_t,t)
=
\mathbb E[\epsilon\mid x_t]
}
$$

不是保证逐样本精确恢复当时加入的随机噪声。

这一点与 NCSN 中的 DSM 完全一致。

### 25.4 DDPM 是否必须使用 ELBO 训练？

从论文的概率模型推导来看，ELBO 是它的重要理论基础。

但实际训练时，作者使用的是：

$$
\boxed{
L_{\mathrm{simple}}
=
\mathbb E
\|\epsilon-\epsilon_\theta(x_t,t)\|^2
}
$$

它不是严格的原始负 ELBO。

因此，应该区分：

- **理论推导：** 从最大似然与变分推断出发。
- **实际训练：** 使用重新加权的噪声预测 MSE。

这也是为什么不能简单地说“DDPM 的噪声预测 MSE 就是 ELBO”。

更准确的说法是：

**DDPM 从 ELBO 推导出加权噪声预测 MSE，再通过经验性地调整权重，得到最终的简化训练目标。**

---

## 二十六、DDPM 与 NCSN 的核心区别

| 对比维度 | NCSN（2019） | DDPM（2020） |
|---|---|---|
| 建模出发点 | Score Matching | 潜变量概率模型 |
| 训练理论 | DSM | ELBO、KL、DSM |
| 前向过程 | 多尺度独立高斯扰动 | 马尔可夫高斯加噪链 |
| 前向形式 | $x=x_0+\sigma\epsilon$ | $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon$ |
| 网络输出 | Score | 噪声 $\epsilon$ |
| 采样方式 | Annealed Langevin Dynamics | 反向马尔可夫链 |
| 采样目标 | 逐级接近不同 $p_\sigma$ 的平稳分布 | 学习从 $x_T$ 到 $x_0$ 的反向转移 |
| 似然建模 | 原始 NCSN 不直接优化 ELBO | 可以通过变分上界评估似然 |
| 核心优势 | 多尺度 Score 学习与退火采样 | 变分推断与去噪训练的统一 |

不过两者有一个深层联系：

$$
\boxed{
\text{DDPM 噪声预测}
\Longleftrightarrow
\text{加噪边缘分布的 Score 预测}
}
$$

因此，DDPM 并不是完全独立于 NCSN 的另一条理论路线。

它将扩散概率模型、变分推断、Score Matching 和类似 Langevin 的随机采样联系起来。

---

## 二十七、整篇论文最值得记住的五个公式

**第一个：前向单步高斯扩散**

$$
\boxed{
q(x_t\mid x_{t-1})
=
\mathcal N
\left(
\sqrt{\alpha_t}x_{t-1},
\beta_tI
\right)
}
$$

**第二个：任意时间步的闭式加噪**

$$
\boxed{
x_t=
\sqrt{\bar\alpha_t}x_0+
\sqrt{1-\bar\alpha_t}\epsilon
}
$$

**第三个：ELBO 的 KL 分解**

$$
\boxed{
\begin{aligned}
L
=
\mathbb E_q\Bigg[
&D_{\mathrm{KL}}(q(x_T\mid x_0)\|p(x_T))\\
&+
\sum_{t=2}^{T}
D_{\mathrm{KL}}
\left(
q(x_{t-1}\mid x_t,x_0)
\|
p_\theta(x_{t-1}\mid x_t)
\right)\\
&-\log p_\theta(x_0\mid x_1)
\Bigg]
\end{aligned}
}
$$

**第四个：简化噪声预测损失**

$$
\boxed{
L_{\mathrm{simple}}
=
\mathbb E_{t,x_0,\epsilon}
\left[
\left\|
\epsilon-
\epsilon_\theta(
\sqrt{\bar\alpha_t}x_0+
\sqrt{1-\bar\alpha_t}\epsilon,t
)
\right\|^2
\right]
}
$$

**第五个：反向采样**

$$
\boxed{
x_{t-1}
=
\frac1{\sqrt{\alpha_t}}
\left(
x_t-
\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}
\epsilon_\theta(x_t,t)
\right)
+
\sigma_tz
}
$$

---

## 二十八、最终总结

DDPM 最重要的理论贡献，是将**概率生成模型的变分推断**与**多尺度去噪学习**建立了明确的数学联系。

它首先构造一个固定的前向马尔可夫链：

$$
q(x_t\mid x_{t-1})
$$

再通过贝叶斯公式推导出：

$$
q(x_{t-1}\mid x_t,x_0)
$$

的解析高斯形式。

随后，从最大化数据对数似然出发，利用 Jensen 不等式构造 ELBO，并将其分解为多个 KL 散度。

由于反向转移采用固定方差的高斯分布，每一步的 KL 可以转化为后验均值 MSE。

再利用前向加噪的重参数化公式，将后验均值表示成噪声的函数，最终得到：

$$
\boxed{
L_{\mathrm{simple}}
=
\mathbb E
\|\epsilon-\epsilon_\theta(x_t,t)\|^2
}
$$

与此同时，噪声预测又与边缘 Score 满足：

$$
\boxed{
\nabla_{x_t}\log q_t(x_t)
=
-\frac{
\mathbb E[\epsilon\mid x_t]
}{
\sqrt{1-\bar\alpha_t}
}
}
$$

因此，DDPM 从变分推断角度得到的训练目标，与 NCSN 从 Score Matching 角度得到的训练目标，在数学上建立了联系。

**如果说 NCSN 的关键是证明“学习 Score 就能够生成数据”，那么 DDPM 的关键就是说明“可以通过优化概率生成模型的变分目标，得到一个简单、稳定的多尺度去噪训练方法，并直接学习反向生成链”。**

最后，还有一个非常值得继续深入的问题：

**既然 DDPM 的噪声预测、$x_0$ 预测和 Score 预测之间都可以通过解析公式相互转换，为什么在实际训练中，预测空间的选择仍然会影响模型效果？**

这个问题会直接联系到你之前研究的 Flow Matching、$x_0$-prediction、不同 Loss 空间，以及 DPM-Solver 的设计。